# Executar todos os notebooks em sequência
Este notebook executa todos os arquivos `.ipynb` na pasta atual (exceto este) em ordem alfabética e salva as versões executadas em `runs/`.

In [ ]:
import os
import glob
import nbformat
from nbclient import NotebookClient

base = os.getcwd()
this_nb = os.path.basename('00_run_all_notebooks.ipynb')
runs_dir = os.path.join(base, 'runs')
os.makedirs(runs_dir, exist_ok=True)

notebooks = sorted([p for p in glob.glob(os.path.join(base, '*.ipynb'))
                    if os.path.basename(p) != this_nb and not os.path.basename(p).startswith('.') and not p.startswith(os.path.join(base, 'runs') + os.sep)])

print(f'Encontrados {len(notebooks)} notebooks para executar:')
for p in notebooks:
    print(' -', os.path.basename(p))

for nb_path in notebooks:
    print('
Executando', nb_path)
    try:
        nb = nbformat.read(nb_path, as_version=4)
        client = NotebookClient(nb, timeout=600, kernel_name='python3')
        client.execute()
        out_path = os.path.join(runs_dir, os.path.splitext(os.path.basename(nb_path))[0] + '_run.ipynb')
        with open(out_path, 'w', encoding='utf-8') as f:
            nbformat.write(nb, f)
        print('Salvo em', out_path)
    except Exception as e:
        print('Erro ao executar', nb_path, ':', e)
        break